# 🧮 Understanding life history dynamics in structured populations with a stage-within-age matrix model

This noteboook implements a structured population model by Diamond *et al*. (1999, 2000)* of a fish called the Atlantic croaker, *Micropogonias undulatus* (for additional information on this species see <a href="http://www.tpwd.state.tx.us/huntwild/wild/species/croaker/.">http://www.tpwd.state.tx.us/huntwild/wild/species/croaker/</a> ). 

![alt](http://www.tpwd.state.tx.us/huntwild/wild/images/fish/acroaker.jpg)

An image of an Atlantic croaker from the [Texas Parks and Wildlife Department](http://www.tpwd.state.tx.us/huntwild/wild/species/croaker).

Key aspects of Atlantic croaker demography, and how Diamond *et al*. formulated their mode of it, are discussed in the [introduction to this Unit](./Croaker.md) and associated pages.  

***Citations**:  
Sandra L. Diamond, Larry B. Crowder, Lindsay G. Cowell (1999). Catch and Bycatch: The Qualitative Effects of Fisheries on Population Vital Rates of Atlantic Croaker. Transactions of the American Fisheries Society; 128(6):1085–1105  
Sandra L Diamond; Lindsay G Cowell; Larry B Crowder (2000). Population effects of shrimp trawl bycatch on Atlantic croaker. Canadian Journal of Fisheries and Aquatic Sciences; 57( 10):2010-21.


In [1]:
# Load modules and set graphics environment
from math import *
import numpy as np
from matplotlib import pyplot as plt
plt.ion();
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [2]:
global A
from croaker import LesliePars, CroakerPopModel

## Exploration of the stage-within-age model: Model Inputs
The input boxes below enable you to experiment with population trajectories for for the Atlantic croaker according to Diamond *et al*.'s stage-within-age model. Three types of parameters appear below:</p>
<p style="margin-left: 30px;">(<em>a</em>) <strong>Simulation parameters</strong>: The input grid in the top row determines the initial population in each Year Class for a population simulation. The fourth input level specifies the number of years to be simulated.&nbsp;</p>
<p style="margin-left: 30px;">(<em>b</em>) <strong>Demographic parameters</strong>: The second, third, fifth and sixth input rows specify the fecundity, mortality and duration parameters. The default values are estimates determined by Diamond et al. to apply to the Atlantic croaker population in the Gulf of Mexico (they also estimated parameters for the South and Mid-Atlantic Bights; see the paper for these parameter values).</p>
<p style="margin-left: 30px;">(<em>c</em>) <strong>Evolutionary parameters</strong>: The bottom input box enables you to specify the Egg Investment Factor, <em>EIF</em>. The default value is 1, representing the observed egg size. <em>EIF</em>&lt;1 implies more, smaller eggs and <em>EIF</em>&gt;1 implies fewer, larger eggs. The parameter $\alpha$ reflects an assumed exponential growth relationship during the Ocean Larva stage. Larger values of $\alpha$ correspond to faster growth rates; smaller values of $\alpha$ correspond to slower growth rates. Both $EIF$ and $\alpha$ affect duration of the Ocean Larva stage.</p>
<p>&nbsp;</p>

<p><span style="font-size: medium;"><strong>6. Exploration of <span style="font-size: medium;"><strong>the stage-within-age model</strong></span>: Model Outputs</strong></span></p>
<p>The model produces four types of output, in the form of plots (numerical data for these plots are also printed out, in case they are useful):</p>
<p style="margin-left: 30px;">(<em>a</em>) <strong>Age distribution</strong>: This plot shows the fraction of the croaker population in each year class. Note that the vertical axis is on a log10 scale. The initial age distribution (set by you in the top input grid) is plotted in green. The stable age distribution, which is the relative fraction of population in each Year Class if the simulation is allowed to run for infinite time, is plotted in blue. The actual final age structure at the end of the simulation is plotted in black.</p>
<p style="margin-left: 30px;">(<em>b</em>) <strong>Population time series</strong>: This plot shows total population of Atlantic croakers, as a function of years from the start of the simulation. The full age-within-stage model population trajectory is shown in green. An analytical approximation to this model, in which it is assumed that the population always matches the stable age distribution, is shown in blue.</p>
<p style="margin-left: 30px;">(<em>c</em>) <strong>Larval demography</strong>: This plot shows the cummulative probability of survival through the successive larval stages. As in Diamond et al.'s paper, time units for larval demography are days (not years, as for the time series plot). Note the vertical axis is on a log10 scale.&nbsp;</p>
<p style="margin-left: 30px;">(<em>d</em>) <strong>Elasticities</strong>: This plot shows the elasticities for each parameter in the simulation. Elasticities are a metric of sensitivity of long-term population increase or decrease to small changes in parameters. For example, a small positive elasticity implies that increase of the corresponding parameter will slightly raise the long-term population trajectory. A large negative elasticity implies an increase in the corresponding parameter will substantially decrease the long-term population trajectory.&nbsp;</p>

In [3]:
# Instantiate a GUI to modify parameters. 
#  Initial parameters from Diamond et al. (2000) for Atlantic croakers in the Gulf of Mexico:
T_egg=widgets.FloatText(value=2.,description = r"$T_{EGG}$")
T_ysl=widgets.FloatText(value=4.,description = r"$T_{YSL}$")
T_ol=widgets.FloatText(value=45.,description = r"$T_{OL}$")
T_el=widgets.FloatText(value=58.5,description = r"$T_{EL}$")
T_ej=widgets.FloatText(value=75.,description = r"$T_{EJ}$")
T_lj=widgets.FloatText(value=180.,description = r"$T_{LJ}$")
EIF=widgets.FloatText(value=1,description = r"$EIF$")

mu_egg=widgets.FloatText(value=0.4984,description = r"$\mu_{EGG}$")
mu_ysl=widgets.FloatText(value=0.1645,description = r"$\mu_{YSL}$")
mu_ol=widgets.FloatText(value=0.09,description = r"$\mu_{OL}$")
mu_el=widgets.FloatText(value=0.0591,description = r"$\mu_{EL}$")
mu_ej=widgets.FloatText(value=0.021,description = r"$\mu_{EJ}$")
mu_lj=widgets.FloatText(value=0.009,description = r"$\mu_{LJ}$")
mu_adult=widgets.FloatText(value=0.85,description = r"$\mu_{adult}$")

F1=widgets.FloatText(value=0.10,description = r"$F_1$")
F2=widgets.FloatText(value=59581.,description = r"$F_2$")
F3=widgets.FloatText(value=76456.,description = r"$F_3$")
F4=widgets.FloatText(value=93690.,description = r"$F_4$")
F5=widgets.FloatText(value=106614.,description = r"$F_5$")
F6=widgets.FloatText(value=124565.,description = r"$F_6$")
F7=widgets.FloatText(value=130310.,description = r"$F_7$")
alpha=widgets.FloatText(value=0.1,description = r"$\alpha$")

p_egg=widgets.FloatText(value=1.,description = r"$p_{egg}$")
p1=widgets.FloatText(value=1.e-12,description = r"$p_1$")
p2=widgets.FloatText(value=1.e-12,description = r"$p_2$")
p3=widgets.FloatText(value=1.e-12,description = r"$p_3$")
p4=widgets.FloatText(value=1.e-12,description = r"$p_4$")
p5=widgets.FloatText(value=1.e-12,description = r"$p_5$")
p6=widgets.FloatText(value=1.e-12,description = r"$p_6$")
p7=widgets.FloatText(value=1.e-12,description = r"$p_7$")
N_years=widgets.IntText(value=7,description = r"$N_{years}$")

ui1 = widgets.VBox([T_egg,T_ysl,T_ol,T_el,T_ej,T_lj,EIF,alpha])
ui2 = widgets.VBox([mu_egg,mu_ysl,mu_ol,mu_el,mu_ej,mu_lj,mu_adult,N_years])
ui3 = widgets.VBox([F1,F2,F3,F4,F5,F6,F7])
ui4 = widgets.VBox([p_egg,p1,p2,p3,p4,p5,p6,p7])
uiC = widgets.HBox([ui1,ui2,ui3,ui4])
    
out = widgets.interactive_output(CroakerPopModel,{'T_egg':T_egg,'T_ysl':T_ysl,'T_ol':T_ol,'T_el':T_el,'T_ej':T_ej,'T_lj':T_lj,
                 'mu_egg':mu_egg,'mu_ysl':mu_ysl,'mu_ol':mu_ol,'mu_el':mu_el,'mu_ej':mu_ej,'mu_lj':mu_lj,
                 'mu_adult':mu_adult,
                    'F1':F1,'F2':F2,'F3':F3,'F4':F4,'F5':F5,'F6':F6,'F7':F7,
                    'p_egg':p_egg,'p1':p1,'p2':p2,'p3':p3,'p4':p4,'p5':p5,'p6':p6,'p7':p7,
                    'N_years':N_years,'alpha':alpha,'EIF':EIF})
display(uiC,out)

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 2000x1200 with 4 Axes>', …

<p><span style="font-size: medium;"><strong>7. Exploration of <span style="font-size: medium;"><strong>the stage-within-age model</strong></span>: Study Questions<br /></strong></span></p>
<p>Your tasks are to play with Diamond et al.'s stage-within-age model for awhile to familiarize yourself with how changes to the simulation, demographic and evolutionary parameters relate to larval survivorship, Year Class structure, and long-term population trends; then, address the following questions:</p>
<p>1) <strong>Population structure</strong>:</p>
<p style="margin-left: 30px;">The key parameters of the model are expressed in the Leslie matrix, <strong>B</strong>. Using results from linear algebra, is can be shown that populations with this form of matrix always approach a stable age distribution. That is, if you run them long enough, the population distribution will approach a steady state (blue). Note that this does not mean the total population is constant, only that the fraction of population in a given Year Class is constant. The steady state is a convenient benchmark for population analysis, because it is easy to calculate using mathematical tools from linear algebra, and it is a characteristic only of the Leslie matrix, not involving simulation-specific parameters like initial population structure and number of years simulated.</p>
<p style="margin-left: 30px;"><strong>In what ways is the stable age distribution informative or uninformative?</strong> Is the analytical steady state age structure a good approximation for the full model? For how many years do initial conditions have significant impacts on age structure? In the event of a catastrophe (e.g., an oil spill), how long do you expect age structure impacts to last? Consider more moderate but continual environmental variations (which are not considered explicitly in this model). What kinds of variations (e.g., annual, decadal, etc.) would you expect to change croaker age structure?</p>
<p style="margin-left: 30px;"><strong>What are the key Year Classes?</strong> Given the relative abundance of fish of different ages, which size classes would be most impacted by targeted fishing or by-catch? Factoring in the age-dependent fecundity parameters, which size classes contribute most to reproduction?</p>
<p>2) <strong>Population time series</strong>:</p>
<p style="margin-left: 30px;">The stable age distribution corresponds to a total population that always changes geometrically in time (i.e., in a linear trajectory on a semi-log plot). The full model does not necessarily follow this type of trajectory.</p>
<p style="margin-left: 30px;"><strong>What do the time series for total population in the full model, and the time series for the stable age distribution (derived from eigenvalue analysis), say about long-term trends in the croaker population?</strong> Do these time series predict long-term increases or decreases? The two time series begin with the same total population; in what ways are the two time series similar or different? How do differences relate to your answers to 1)?&nbsp;</p>
<p>3) <strong>Larval demography</strong>:</p>
<p style="margin-left: 30px;"><strong>What do probabilities of larval survival across stages suggest about vunerability during early development?</strong> On average, how many eggs must a female lay to produce one offspring that survives its first year? What stages account for the most significant mortality? Which larval characteristics (e.g., size, sensory capabilities, swimming abilities, habitat, etc.) do you hypothesize might be most responsible for differences? In what ways would a survival probabilities differ in a fish species that gives live birth to a few, more-developed larvae? If you were sampling the plankton to assess croaker reproduction, how would estimates of larval survival inform your sampling design?</p>
<p>4) <strong>Elasticities</strong>:</p>
<p style="margin-left: 30px;"><strong>What do elasticities for key parameters suggest about the croaker population's sensitivity to demographic parameters?</strong> Some parameters are inter-related. For example, in feeding larval stages, behaviors that enhance feeding rates often incur added risk of predation. Hence, a change in foraging behavior that increases feeding might both&nbsp;shorten duration of a larval stage (T) and increase mortality rates (mu). What determines whether this is a beneficial or harmful change?</p>
<p>5) <strong>Evolution of egg size</strong>:</p>
<p style="margin-left: 30px;">Keep in mind that the egg-size model implemented here is oversimplified in important ways. Nonetheless, it may suffice to suggest whether, under the fishing pressure and demographic regime modeled by Diamond et al., different egg sizes might evolve and, if so, whether eggs are likely to become bigger or smaller.&nbsp;</p>
<p style="margin-left: 30px;"><strong>What are the long-term consequences of changes in egg size?</strong> Consider what magnitude of change might be relatively "easy" -- that is, not require major reorganization of egg structure or larval development mechanisms (clearly a speculative threshold...).&nbsp; Within a spectrum of such changes, would you expect to see evolutionary changes in croaker egg size?</p>
<p>6) <strong>Responses to climate change</strong>:</p>
<p style="margin-left: 30px;">Consider one of the many factors predicted to change in future oceans (temperature, pH, mixed layer depth, nutrient levels, alien invasions, pollution, human exploitation, etc.). Speculate how this factor might affect parameters like larval duration and mortality rates. Use insights from the model to predict the impacts (or lack of impacts) on future croaker population dynamics.&nbsp; </p>